# 01_Paper_Full_Cuda

本 Notebook 用于在 **Google Colab** 中从零启动 `CEG-WM` 项目正式工作流，并将项目结果持续化保存到 `Google Drive/MyDrive/CEG_WM_Outputs_project_root` 目录下。

## 目标

1. 挂载 `Google Drive` 并建立持久化输出根目录。  
2. 从 GitHub 拉取 `CEG-WM` 仓库代码。  
3. 安装项目依赖。  
4. 下载并缓存 `Stable Diffusion 3.5` 模型与 `InSPyReNet` 权重。  
5. 使用 **`configs/default.yaml` 作为唯一默认运行配置**。  
6. 在运行前执行 formal GPU 前置环境校验。  
7. 调用 `notebook/01_Paper_Full_Cuda.py` 执行具体编排。  
8. 将 `run_manifest`、主 `run_root`、日志、摘要与导出结果持久化到 `CEG_WM_Outputs_project_root` 下。

## 运行约束

1. 本 Notebook 仅承担运行入口与环境准备职责。  
2. 具体编排逻辑应收敛到 `notebook/01_Paper_Full_Cuda.py`。  
3. 本 Notebook 不再直接以散落的 shell 命令拼装 embed / detect / calibrate / evaluate。  
4. 本 Notebook 假定仓库中的 `configs/default.yaml` 已经承载正式默认配置。  
5. 若 `notebook/01_Paper_Full_Cuda.py` 尚未构建，本 Notebook 会在调用前明确报错。

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_NAME = "01_Paper_Full_Cuda"

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "main"
REPO_DIR_NAME = "CEG-WM"

COLAB_WORKDIR = Path("/content/ceg_wm_workspace")
REPO_ROOT = COLAB_WORKDIR / REPO_DIR_NAME

DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_Outputs_project_root"
DRIVE_RUNS_ROOT = DRIVE_PROJECT_ROOT / "runs" / NOTEBOOK_NAME
DRIVE_RUNTIME_STATE_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME
DRIVE_EXPORT_ROOT = DRIVE_PROJECT_ROOT / "exports" / NOTEBOOK_NAME
DRIVE_LOG_ROOT = DRIVE_PROJECT_ROOT / "logs" / NOTEBOOK_NAME
DRIVE_SECRETS_ROOT = DRIVE_PROJECT_ROOT / "secrets"

CONFIG_RELATIVE_PATH = "configs/default.yaml"
ORCHESTRATOR_RELATIVE_PATH = "notebook/01_Paper_Full_Cuda.py"
MODEL_CACHE_DIR = COLAB_WORKDIR / "huggingface_cache"
LOCAL_MODEL_WEIGHT_ROOT = COLAB_WORKDIR / "models"
INSPYRENET_WEIGHT_RELATIVE_PATH = Path("models/inspyrenet/ckpt_base.pth")
INSPYRENET_DEFAULT_URL = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"
ATTESTATION_ENV_FILE = DRIVE_SECRETS_ROOT / "attestation_env.json"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REQUEST_RUN_ROOT = DRIVE_RUNS_ROOT / RUN_TAG
WORKSPACE_SUMMARY_PATH = DRIVE_RUNTIME_STATE_ROOT / "workspace_summary.json"

ALLOW_EPHEMERAL_ATTESTATION_KEYS = False

print("=" * 100)
print("01_Paper_Full_Cuda 参数初始化完成")
print("=" * 100)
print(f"REPO_URL: {REPO_URL}")
print(f"REPO_BRANCH: {REPO_BRANCH}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"CONFIG_RELATIVE_PATH: {CONFIG_RELATIVE_PATH}")
print(f"ORCHESTRATOR_RELATIVE_PATH: {ORCHESTRATOR_RELATIVE_PATH}")
print(f"DRIVE_PROJECT_ROOT: {DRIVE_PROJECT_ROOT}")
print(f"REQUEST_RUN_ROOT: {REQUEST_RUN_ROOT}")

In [ ]:
def run_checked(command, cwd=None, env=None, label=None):
    if label:
        print(f"\n[RUN] {label}")
    print(" ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-4000:])
        raise RuntimeError(
            f"命令执行失败，return_code={result.returncode}，command={' '.join(command)}"
        )
    return result


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding="utf-8"))
    return payload if isinstance(payload, dict) else {}


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def maybe_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() == dst.resolve():
        return
    shutil.copy2(src, dst)


print("Notebook 工具函数已就绪。")

## 第 1 步：挂载 `Google Drive` 并建立持久化目录

In [ ]:
try:
    from google.colab import drive
except Exception as exc:
    raise RuntimeError("本 Notebook 设计为在 Google Colab 中运行。") from exc

drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)

for path in [
    COLAB_WORKDIR,
    DRIVE_PROJECT_ROOT,
    DRIVE_RUNS_ROOT,
    DRIVE_RUNTIME_STATE_ROOT,
    DRIVE_EXPORT_ROOT,
    DRIVE_LOG_ROOT,
    DRIVE_SECRETS_ROOT,
]:
    ensure_dir(path)

workspace_bootstrap = {
    "notebook_name": NOTEBOOK_NAME,
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_root": str(REPO_ROOT),
    "config_relative_path": CONFIG_RELATIVE_PATH,
    "orchestrator_relative_path": ORCHESTRATOR_RELATIVE_PATH,
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "request_run_root": str(REQUEST_RUN_ROOT),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(WORKSPACE_SUMMARY_PATH, workspace_bootstrap)

print("✓ Google Drive 已挂载")
print(f"✓ 项目持久化根目录：{DRIVE_PROJECT_ROOT}")
print(f"✓ workspace_summary: {WORKSPACE_SUMMARY_PATH}")

## 第 2 步：执行运行前环境预检

In [ ]:
import psutil

print("=" * 100)
print("系统环境预检")
print("=" * 100)

python_version = sys.version.replace("\n", " ")
platform_text = platform.platform()
cpu_count = os.cpu_count()
memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)
disk_usage = shutil.disk_usage("/content")
disk_free_gb = round(disk_usage.free / (1024 ** 3), 2)

print(f"Python: {python_version}")
print(f"Platform: {platform_text}")
print(f"CPU count: {cpu_count}")
print(f"RAM: {memory_gb} GiB")
print(f"Disk free: {disk_free_gb} GiB")

nvidia_smi_path = shutil.which("nvidia-smi")
if nvidia_smi_path is None:
    raise RuntimeError("未检测到 nvidia-smi，当前环境不满足 formal GPU 运行前提。")

gpu_result = run_checked(["nvidia-smi"], label="GPU 信息检查")
gpu_report_path = DRIVE_RUNTIME_STATE_ROOT / "nvidia_smi.txt"
gpu_report_path.write_text(gpu_result.stdout, encoding="utf-8")

print(f"✓ GPU 工具可用：{nvidia_smi_path}")
print(f"✓ GPU 报告已写入：{gpu_report_path}")

## 第 3 步：同步仓库代码

In [ ]:
print("=" * 100)
print("同步 GitHub 仓库")
print("=" * 100)

if shutil.which("git") is None:
    raise RuntimeError("当前环境未检测到 git 命令。")

ensure_dir(COLAB_WORKDIR)

repo_exists = REPO_ROOT.exists() and (REPO_ROOT / ".git").exists()
sync_mode = "git_clone"

if repo_exists:
    sync_mode = "git_fetch_reset"
    run_checked(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH, "--depth", "1"], label="git fetch")
    run_checked(["git", "-C", str(REPO_ROOT), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], label="git checkout")
    run_checked(["git", "-C", str(REPO_ROOT), "reset", "--hard", "FETCH_HEAD"], label="git reset")
    run_checked(["git", "-C", str(REPO_ROOT), "clean", "-fd"], label="git clean")
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
        label="git clone",
    )

required_repo_paths = [
    REPO_ROOT / "configs" / "default.yaml",
    REPO_ROOT / "scripts",
    REPO_ROOT / "main",
    REPO_ROOT / "requirements.txt",
]
missing_repo_paths = [str(path) for path in required_repo_paths if not path.exists()]
if missing_repo_paths:
    raise FileNotFoundError(f"仓库同步后缺少关键路径：{missing_repo_paths}")

git_info = {
    "sync_mode": sync_mode,
    "repo_root": str(REPO_ROOT),
    "repo_url": run_checked(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], label="git remote").stdout.strip(),
    "branch": run_checked(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"], label="git branch").stdout.strip(),
    "commit_hash": run_checked(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], label="git rev-parse").stdout.strip(),
    "commit_hash_short": run_checked(["git", "-C", str(REPO_ROOT), "rev-parse", "--short=8", "HEAD"], label="git rev-parse short").stdout.strip(),
    "commit_message": run_checked(["git", "-C", str(REPO_ROOT), "log", "-1", "--pretty=%s"], label="git log subject").stdout.strip(),
    "commit_timestamp": run_checked(["git", "-C", str(REPO_ROOT), "log", "-1", "--pretty=%ci"], label="git log timestamp").stdout.strip(),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
git_info_path = DRIVE_RUNTIME_STATE_ROOT / "git_info.json"
write_json(git_info_path, git_info)

print(f"✓ 仓库已就绪：{REPO_ROOT}")
print(f"✓ Git 信息：{git_info_path}")

## 第 4 步：安装项目依赖

In [ ]:
print("=" * 100)
print("安装项目依赖")
print("=" * 100)

pip_install_log = DRIVE_LOG_ROOT / f"{RUN_TAG}_pip_install.log"

install_commands = [
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)],
]

log_chunks = []
for command in install_commands:
    result = subprocess.run(
        command,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    log_chunks.append(
        "\n".join(
            [
                "=" * 80,
                "COMMAND:",
                " ".join(command),
                "RETURN_CODE:",
                str(result.returncode),
                "STDOUT:",
                result.stdout or "",
                "STDERR:",
                result.stderr or "",
            ]
        )
    )
    if result.returncode != 0:
        pip_install_log.write_text("\n\n".join(log_chunks), encoding="utf-8")
        raise RuntimeError(
            f"依赖安装失败，command={' '.join(command)}，详见 {pip_install_log}"
        )

pip_install_log.write_text("\n\n".join(log_chunks), encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"✓ 依赖安装完成，日志：{pip_install_log}")

## 第 5 步：校验 `default.yaml` 作为唯一默认运行配置的前提

In [ ]:
import yaml

default_cfg_path = REPO_ROOT / CONFIG_RELATIVE_PATH
paper_cfg_path = REPO_ROOT / "configs" / "paper_full_cuda.yaml"

default_cfg = yaml.safe_load(default_cfg_path.read_text(encoding="utf-8"))
paper_cfg = yaml.safe_load(paper_cfg_path.read_text(encoding="utf-8")) if paper_cfg_path.exists() else None

if not isinstance(default_cfg, dict):
    raise TypeError("configs/default.yaml 根节点必须为 mapping。")

if paper_cfg is not None and default_cfg != paper_cfg:
    raise RuntimeError(
        "当前 notebook 假定 configs/default.yaml 与 configs/paper_full_cuda.yaml 语义一致；"
        "但仓库中的两者已不一致，请先收敛唯一默认配置后再运行。"
    )

matrix_cfg = default_cfg.get("experiment_matrix")
matrix_config_path = None
if isinstance(matrix_cfg, dict):
    matrix_config_path = matrix_cfg.get("config_path")

cfg_guard_summary = {
    "default_cfg_path": str(default_cfg_path),
    "paper_cfg_path": str(paper_cfg_path),
    "default_equals_paper_full": bool(paper_cfg is not None and default_cfg == paper_cfg),
    "experiment_matrix_config_path": matrix_config_path,
    "device": default_cfg.get("device"),
    "model_id": default_cfg.get("model_id"),
    "inference_prompt_file": default_cfg.get("inference_prompt_file"),
    "parallel_attestation_statistics_enabled": bool(
        isinstance(default_cfg.get("parallel_attestation_statistics"), dict)
        and default_cfg.get("parallel_attestation_statistics", {}).get("enabled", False)
    ),
}
cfg_guard_summary_path = DRIVE_RUNTIME_STATE_ROOT / "default_config_guard_summary.json"
write_json(cfg_guard_summary_path, cfg_guard_summary)

print("✓ default.yaml 加载成功")
print(json.dumps(cfg_guard_summary, indent=2, ensure_ascii=False))

## 第 6 步：下载并缓存模型与权重

In [ ]:
import gc
from huggingface_hub import scan_cache_dir, snapshot_download

print("=" * 100)
print("下载并缓存 Stable Diffusion 3.5 模型")
print("=" * 100)

ensure_dir(MODEL_CACHE_DIR)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

model_id = str(default_cfg.get("model_id", "stabilityai/stable-diffusion-3.5-medium"))

def is_model_cached(repo_id: str) -> bool:
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if repo_id == repo.repo_id:
                return True
        return False
    except Exception:
        return False

if is_model_cached(model_id):
    print(f"✓ 模型已缓存：{model_id}")
else:
    print(f"开始下载：{model_id}")
    snapshot_download(
        repo_id=model_id,
        cache_dir=str(MODEL_CACHE_DIR),
        local_files_only=False,
    )
    print("✓ 模型下载完成")

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

print(f"✓ 模型缓存目录：{MODEL_CACHE_DIR}")

In [ ]:
import urllib.request

print("=" * 100)
print("下载并缓存 InSPyReNet 权重")
print("=" * 100)

repo_weight_path = REPO_ROOT / INSPYRENET_WEIGHT_RELATIVE_PATH
cached_weight_path = LOCAL_MODEL_WEIGHT_ROOT / "inspyrenet" / "ckpt_base.pth"
ensure_dir(cached_weight_path.parent)
ensure_dir(repo_weight_path.parent)

candidate_paths = [repo_weight_path, cached_weight_path]
existing_weight = None
for candidate in candidate_paths:
    if candidate.exists() and candidate.is_file() and candidate.stat().st_size > 0:
        existing_weight = candidate
        break

if existing_weight is None:
    temp_path = cached_weight_path.with_suffix(".pth.downloading")
    if temp_path.exists():
        temp_path.unlink()
    urllib.request.urlretrieve(INSPYRENET_DEFAULT_URL, str(temp_path))
    if not temp_path.exists() or temp_path.stat().st_size <= 0:
        raise RuntimeError("InSPyReNet 权重下载失败或文件为空。")
    temp_path.replace(cached_weight_path)
    existing_weight = cached_weight_path
    print(f"✓ 已下载：{existing_weight}")
else:
    print(f"✓ 已复用现有权重：{existing_weight}")

maybe_copy(existing_weight, cached_weight_path)
maybe_copy(existing_weight, repo_weight_path)

weight_summary = {
    "repo_weight_path": str(repo_weight_path),
    "cached_weight_path": str(cached_weight_path),
    "selected_weight_path": str(repo_weight_path if repo_weight_path.exists() else cached_weight_path),
    "size_bytes": (repo_weight_path if repo_weight_path.exists() else cached_weight_path).stat().st_size,
}
weight_summary_path = DRIVE_RUNTIME_STATE_ROOT / "inspyrenet_weight_summary.json"
write_json(weight_summary_path, weight_summary)

print(json.dumps(weight_summary, indent=2, ensure_ascii=False))

## 第 7 步：加载 attestation 环境变量并执行 formal GPU 前置校验

In [ ]:
attestation_env = load_json(ATTESTATION_ENV_FILE)

required_attestation_env_names = []
attestation_cfg = default_cfg.get("attestation", {})
if isinstance(attestation_cfg, dict):
    for key_name in ("k_master_env_var", "k_prompt_env_var", "k_seed_env_var"):
        value = attestation_cfg.get(key_name)
        if isinstance(value, str) and value:
            required_attestation_env_names.append(value)

if attestation_env:
    for key_name, value in attestation_env.items():
        if isinstance(key_name, str) and isinstance(value, str) and value:
            os.environ[key_name] = value

if ALLOW_EPHEMERAL_ATTESTATION_KEYS:
    import secrets
    for env_name in required_attestation_env_names:
        if not os.environ.get(env_name):
            os.environ[env_name] = secrets.token_hex(32)

missing_attestation_env = [name for name in required_attestation_env_names if not os.environ.get(name)]
if missing_attestation_env:
    raise RuntimeError(
        "缺少必需的 attestation 环境变量。"
        f"请在 {ATTESTATION_ENV_FILE} 中提供，缺失项：{missing_attestation_env}"
    )

from scripts.workflow_acceptance_common import detect_formal_gpu_preflight

preflight = detect_formal_gpu_preflight(default_cfg_path)
preflight_path = DRIVE_RUNTIME_STATE_ROOT / "formal_gpu_preflight.json"
write_json(preflight_path, preflight)

if not bool(preflight.get("ok", False)):
    raise RuntimeError(f"formal GPU 前置校验失败：{json.dumps(preflight, ensure_ascii=False)}")

print("✓ attestation 环境变量已就绪")
print(json.dumps(preflight, indent=2, ensure_ascii=False))

## 第 8 步：调用 `01_Paper_Full_Cuda.py` 执行正式项目流程

In [ ]:
orchestrator_path = REPO_ROOT / ORCHESTRATOR_RELATIVE_PATH
if not orchestrator_path.exists():
    raise FileNotFoundError(
        f"未找到编排脚本：{orchestrator_path}。请先根据配套 Codex Prompt 构建该文件。"
    )

run_request = {
    "notebook_name": NOTEBOOK_NAME,
    "repo_root": str(REPO_ROOT),
    "config_path": str(default_cfg_path),
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "request_run_root": str(REQUEST_RUN_ROOT),
    "drive_export_root": str(DRIVE_EXPORT_ROOT),
    "drive_log_root": str(DRIVE_LOG_ROOT),
    "attestation_env_file": str(ATTESTATION_ENV_FILE),
    "workspace_summary_path": str(WORKSPACE_SUMMARY_PATH),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
run_request_path = DRIVE_RUNTIME_STATE_ROOT / f"{RUN_TAG}_run_request.json"
write_json(run_request_path, run_request)

orchestrator_cmd = [
    sys.executable,
    str(orchestrator_path),
    "--repo-root", str(REPO_ROOT),
    "--config", str(default_cfg_path),
    "--drive-project-root", str(DRIVE_PROJECT_ROOT),
    "--request-run-root", str(REQUEST_RUN_ROOT),
    "--drive-export-root", str(DRIVE_EXPORT_ROOT),
    "--drive-log-root", str(DRIVE_LOG_ROOT),
    "--workspace-summary-path", str(WORKSPACE_SUMMARY_PATH),
    "--run-request-path", str(run_request_path),
]

print("将执行：")
print(" ".join(orchestrator_cmd))

orchestrator_result = subprocess.run(
    orchestrator_cmd,
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

orchestrator_stdout_path = DRIVE_LOG_ROOT / f"{RUN_TAG}_01_Paper_Full_Cuda_stdout.log"
orchestrator_stderr_path = DRIVE_LOG_ROOT / f"{RUN_TAG}_01_Paper_Full_Cuda_stderr.log"
orchestrator_stdout_path.write_text(orchestrator_result.stdout or "", encoding="utf-8")
orchestrator_stderr_path.write_text(orchestrator_result.stderr or "", encoding="utf-8")

print(orchestrator_result.stdout[-8000:] if orchestrator_result.stdout else "<empty stdout>")
if orchestrator_result.returncode != 0:
    print(orchestrator_result.stderr[-8000:] if orchestrator_result.stderr else "<empty stderr>")
    raise RuntimeError(
        f"01_Paper_Full_Cuda.py 执行失败，return_code={orchestrator_result.returncode}。"
        f"stdout={orchestrator_stdout_path}，stderr={orchestrator_stderr_path}"
    )

print(f"✓ orchestrator stdout: {orchestrator_stdout_path}")
print(f"✓ orchestrator stderr: {orchestrator_stderr_path}")

## 第 9 步：读取 run manifest 并核验结果落盘

In [ ]:
manifest_candidates = sorted(DRIVE_RUNTIME_STATE_ROOT.glob(f"{RUN_TAG}*_run_manifest.json"))
if not manifest_candidates:
    manifest_candidates = sorted(DRIVE_RUNTIME_STATE_ROOT.glob("*run_manifest.json"))

if not manifest_candidates:
    raise FileNotFoundError(
        "未找到 run_manifest.json。请检查 01_Paper_Full_Cuda.py 是否按约定写出 manifest。"
    )

run_manifest_path = manifest_candidates[-1]
run_manifest = load_json(run_manifest_path)

print("=" * 100)
print("run_manifest")
print("=" * 100)
print(json.dumps(run_manifest, indent=2, ensure_ascii=False))

resolved_run_root = Path(run_manifest.get("resolved_run_root", REQUEST_RUN_ROOT))
required_paths = [
    resolved_run_root / "records" / "embed_record.json",
    resolved_run_root / "records" / "detect_record.json",
    resolved_run_root / "records" / "calibration_record.json",
    resolved_run_root / "records" / "evaluate_record.json",
    resolved_run_root / "artifacts" / "evaluation_report.json",
    resolved_run_root / "artifacts" / "run_closure.json",
    resolved_run_root / "artifacts" / "thresholds" / "thresholds_artifact.json",
]

missing_required = [str(path) for path in required_paths if not path.exists()]
if missing_required:
    raise RuntimeError(f"主结果目录缺少必需产物：{missing_required}")

print("\n✓ 主结果目录已落盘且必需产物齐全")
print(f"✓ resolved_run_root: {resolved_run_root}")
print(f"✓ run_manifest_path: {run_manifest_path}")

## 第 10 步：读取最终摘要并确认 Google Drive 持久化状态

In [ ]:
summary_candidates = [
    resolved_run_root / "artifacts" / "workflow_acceptance" / "paper_full_formal_summary.json",
    resolved_run_root / "artifacts" / "workflow_summary.json",
]
summary_payload = {}
summary_path = None
for candidate in summary_candidates:
    if candidate.exists():
        summary_payload = load_json(candidate)
        summary_path = candidate
        break

final_report = {
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "resolved_run_root": str(resolved_run_root),
    "run_manifest_path": str(run_manifest_path),
    "summary_path": str(summary_path) if summary_path is not None else "<absent>",
    "summary_exists": summary_path is not None and summary_path.exists(),
    "workflow_status": summary_payload.get("workflow_status", summary_payload.get("status", "<absent>")) if isinstance(summary_payload, dict) else "<absent>",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
final_report_path = DRIVE_RUNTIME_STATE_ROOT / f"{RUN_TAG}_final_report.json"
write_json(final_report_path, final_report)

print("=" * 100)
print("最终结论")
print("=" * 100)
print(json.dumps(final_report, indent=2, ensure_ascii=False))
print(f"✓ final_report_path: {final_report_path}")